# 29c · **Toàn bộ** vi khuẩn — bỏ `top-K`, chuyển sang `EmbeddingBag`

## Đang bỏ đi cái gì

`29b` giữ **30** organism:

```python
ORG_TOP_K = 30
ORG_VOCAB = _tr.org_itemid.value_counts().head(ORG_TOP_K).index.tolist()
micro[f'micro_org_{oid}'] = ...   # 30 cot, gia tri = SO LAN moc (khong phai 0/1)
```

Ngoài top-30, **danh tính mất hẳn**; chỉ `micro_pos_n` và `micro_n_org` (đếm) sống sót.

## Vì sao "lấy hết" **bắt buộc** đổi biểu diễn

Giữ one-hot mà mở hết từ vựng thì hỏng ở hai chỗ độc lập:

```
EHRColTok  phat MOT token moi cot -> ~V token thay vi 248, va cot vang mat VAN phat token b_j
chuan hoa  cot cuc hiem bi z-score ra gia tri khong lo (p = 1/219,785 -> 468)
           37 da phai va bang "cot nhi phan giu 0/1" — nhung do la VA, khong phai sua
```

Dạng đúng là **danh sách chỉ số + `nn.EmbeddingBag`**:

```
luu   micro_org_ids = [12, 447, 1058]   micro_org_cnt = [1.0, 2.0, 1.0]
doc   EmbeddingBag(V, d, mode='sum', per_sample_weights=cnt)
```

`O(số vi khuẩn CÓ MẶT)`, không phải `O(V)`. Không còn cột chỉ báo nào để chia cho độ lệch chuẩn ⇒
lỗi `468` **chết tận gốc** thay vì bị vá. Một vi khuẩn gặp 3 lần tốn `d` tham số và nằm yên gần
điểm khởi tạo — nó không bóp méo chuẩn hoá của bất kỳ cột nào khác.

> **Không lập thùng `<rare>`** — đó là `top-K` đội tên khác. Giữ **mọi** vi khuẩn làm ID riêng.
> Chỉ có `<unk>` (id 0) cho organism xuất hiện ở val/test mà **không có trong train**, giữ đúng
> kỷ luật *"vocab khớp CHỈ trên train"* mà `29b` đã đặt.

## Điều này KHÔNG hứa gì về mAP

`06` đo *"mở rộng micro / vitals / comorbidity"* = **`−0.0002`, p=0.901** trên cohort cũ. Nhưng phép
mở rộng đó là **thêm cột one-hot**, không phải **mở từ vựng + gộp bag** — khác phép thử.

Tiên nghiệm yếu nhưng không thù địch: `33` C7 tìm ra nhóm nhãn nhạy EHR là **nhiễm trùng + chấn
thương** (Tuberculosis · Pneumonia · Rib Fracture), tức đúng chỗ mà **danh tính vi khuẩn** đáng ra
phải mang thông tin. Nếu đuôi từ vựng có giá trị thì nó phải hiện ra ở đó, không phải ở macro.

`C7` đo thẳng **top-30 đã vứt đi bao nhiêu** trước khi ai kịp kỳ vọng.

## Không ghi đè gì

```
DOC   cxrlt_base.parquet                 (29)
GHI   cxrlt_micro_full.parquet   MOI     ragged ids + counts, khoa study_id
      cxrlt_micro_vocab.json     MOI     id -> org_name, khop tren TRAIN
KHONG dung  cxrlt_ehr.parquet / cxrlt_ehr_spec.json
```

⇒ `38b` đang chạy **không bị ảnh hưởng**.

## C0 · Config — quy ước giữ nguyên `29b`

In [1]:
import os, json, time, re
import numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'
TMP_DS  = 'cxrlt_tmp'

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
P_BASE  = os.path.join(OUT_DIR, 'cxrlt_base.parquet')
assert os.path.exists(P_BASE), f'khong thay {P_BASE} -> chay 29_cxrlt_ehr_base.ipynb truoc'

# --- cua so GIU Y NGUYEN 29b, neu khong thi khong so duoc voi top-30 ---
MICRO_PAST_D   = 7
MICRO_FUTURE_H = 72
ORG_TOP_K_REF  = 30      # CHI de doi chieu o C7 — 29c KHONG cat

P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')     # FILE MOI
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')       # FILE MOI
# 29c chi GHI vao hai duong dan tren. cxrlt_ehr.parquet KHONG bi dung toi ->
# 38b dang chay khong bi anh huong.
assert P_MICRO not in (os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'),)

BASE = pd.read_parquet(P_BASE)
print(f'{len(BASE):,} study | setting {BASE.setting.value_counts().to_dict()}')
assert BASE.study_id.is_unique and BASE.index_time.notna().all()
print(f'cua so vi sinh: [-{MICRO_PAST_D}d, +{MICRO_FUTURE_H}h] quanh index_time (y het 29b)')
print(f'ghi ra: {P_MICRO}\n        {P_VOCAB}')

Mounted at /content/drive
227,835 study | setting {'inpatient': 110892, 'ed': 77801, 'outpatient': 39142}
cua so vi sinh: [-7d, +72h] quanh index_time (y het 29b)
ghi ra: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_micro_full.parquet
        /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_micro_vocab.json


## C1 · Auth — `gcloud`, **không bao giờ** `authenticate_user`

In [2]:
!gcloud auth application-default login --no-launch-browser

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=azybWVFFiUx5FdpKrBY2m6x4PUNlqr&prompt=consent&token_usage=remote&access_type=offline&code_challenge=px_rjUZSTSLJRjpubDg651CTGYFReDJjksEd6Ky4_Kg&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AXEQxID9O2b-i_iXlGKTivWfailJEsnx6XZk3rpur7KsvfL8pgZaLYjK31LHvjEnBnmDDw

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

## C2 · Client + `run_q` — verbatim `29b`

In [3]:
!gcloud config set project {PROJECT}
!gcloud auth application-default set-quota-project {PROJECT} 2>/dev/null || true

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')

def run_q(sql, label='', poll=2.0):
    '''Dry-run truoc (0 byte tinh tien, bat loi go phim), roi moi chay that.'''
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    est = dry.total_bytes_processed / 1e9
    print(f'  [{label}] uoc quet {est:.2f} GB, bat dau...')
    t0 = time.time(); job = client.query(sql)
    while not job.done():
        done_gb = (job.total_bytes_processed or 0) / 1e9
        pct = f'{done_gb/est:5.0%}' if est > 0 else '  n/a'
        print(f'  [{label}] {job.state:<7} {pct} ({done_gb:5.2f}/{est:.2f} GB) '
              f'· {time.time()-t0:5.0f}s', end='\r')
        time.sleep(poll)
    t_run = time.time() - t0
    try:    df = job.to_dataframe(progress_bar_type='tqdm')
    except Exception: df = job.to_dataframe()
    act = (job.total_bytes_processed or 0) / 1e9
    print(f'  [{label}] {act:5.2f} GB thuc · chay {t_run:5.0f}s · tai '
          f'{time.time()-t0-t_run:4.0f}s · {len(df):>9,} dong')
    return df

print('BigQuery OK ->', [dict(r) for r in client.query('SELECT 1 AS ok').result()])


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].
BigQuery OK -> [{'ok': 1}]


## C3 · Đẩy cohort lên bảng tạm — rút gọn `29b` C3 (vi sinh không cần `ED`)

In [4]:
ds = f'{PROJECT}.{TMP_DS}'
client.create_dataset(bigquery.Dataset(ds), exists_ok=True)
COH = f'`{ds}.coh_micro`'
keys = BASE[['study_id', 'subject_id', 'index_time', 'split']].copy()
keys['index_time'] = pd.to_datetime(keys.index_time)
client.load_table_from_dataframe(keys, f'{ds}.coh_micro',
    job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
_n = int(client.query(f'SELECT COUNT(*) n FROM {COH}').to_dataframe().n[0])
assert _n == len(keys), f'BigQuery {_n} dong vs local {len(keys)}'
TRAIN_IDS = set(BASE.loc[BASE.split == 'train', 'study_id'])
print(f'da day -> {ds}.coh_micro ({_n:,} study) | study train {len(TRAIN_IDS):,}')

da day -> mimic-ehr-v310.cxrlt_tmp.coh_micro (227,835 study) | study train 155,823


## C4 · Kéo **mọi** dòng vi sinh — không cắt gì

Cùng cửa sổ, cùng bất biến chống rò rỉ như `29b` C10. Khác đúng một chỗ: **không `head(K)`**.

In [5]:
_hi = ('c.index_time' if MICRO_FUTURE_H == 0 else
       f'TIMESTAMP_ADD(c.index_time, INTERVAL {MICRO_FUTURE_H} HOUR)')
res = run_q(f'''
SELECT c.study_id, c.split, m.org_itemid,
       ANY_VALUE(m.org_name) AS org_name,
       COUNT(*)              AS n_moc
FROM {COH} c JOIN `{HOSP}.microbiologyevents` m ON m.subject_id = c.subject_id
WHERE CAST(m.charttime AS TIMESTAMP)
        BETWEEN TIMESTAMP_SUB(c.index_time, INTERVAL {MICRO_PAST_D} DAY) AND {_hi}
  AND m.org_itemid IS NOT NULL
GROUP BY c.study_id, c.split, m.org_itemid
''', 'micro-full')

print(f'\n{len(res):,} cap (study, organism) | {res.study_id.nunique():,} study co it nhat 1 organism')
print(f'{res.org_itemid.nunique():,} organism PHAN BIET tren toan bo cohort')

  [micro-full] uoc quet 0.12 GB, bat dau...
Job ID 173ff87c-0799-48b5-95fe-168a1e9eabb9 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
  [micro-full]  0.12 GB thuc · chay     1s · tai    2s ·   106,114 dong

106,114 cap (study, organism) | 63,293 study co it nhat 1 organism
346 organism PHAN BIET tren toan bo cohort


## C5 · Từ vựng khớp **chỉ trên train**, `<unk>` = id 0

`id 0` dành cho organism chỉ xuất hiện ở val/test. Không gộp organism hiếm của train vào `<unk>` —
đó chính là `top-K` đội tên khác, và là thứ ta đang bỏ.

In [6]:
_tr = res[res.study_id.isin(TRAIN_IDS)]
ORG_ALL = _tr.groupby('org_itemid').agg(n_study=('study_id', 'nunique'),
                                        name=('org_name', 'first')).sort_values('n_study', ascending=False)
V = len(ORG_ALL) + 1                                  # +1 cho <unk>
ID_OF = {int(o): i + 1 for i, o in enumerate(ORG_ALL.index)}   # id 0 = <unk>
VOCAB = {'0': '<unk>'} | {str(i): str(ORG_ALL.name.iloc[i-1]) for i in range(1, V)}
print(f'tu vung: {V:,} (gom <unk>) khop tren {len(_tr):,} cap TRAIN')
print(f'so voi 29b: {V-1:,} organism thay vi {ORG_TOP_K_REF}  ->  {(V-1)/ORG_TOP_K_REF:.1f}x')

print('\n=== DUOI TU VUNG: bao nhieu organism hiem den muc nao (tren TRAIN) ===')
for lo, hi in ((1, 1), (2, 4), (5, 19), (20, 99), (100, 10**9)):
    _m = ORG_ALL.n_study.between(lo, hi)
    print(f'  xuat hien o {lo:>4}-{hi if hi < 10**9 else "..":>4} study: {int(_m.sum()):>4} organism'
          f'  ({int(ORG_ALL.n_study[_m].sum()):>8,} cap)')

res['oid'] = res.org_itemid.map(ID_OF).fillna(0).astype('int32')
_unk = int((res.oid == 0).sum())
print(f'\ncap roi vao <unk> (organism khong co trong train): {_unk:,} ({_unk/len(res):.3%})')

g = res.groupby('study_id')
MICRO = pd.DataFrame({
    'study_id':      g.size().index,
    'micro_org_ids': g.oid.apply(list).values,
    'micro_org_cnt': g.n_moc.apply(lambda s: [float(x) for x in s]).values,
    'micro_n_org_full': g.size().values,
}).reset_index(drop=True)
# study KHONG co ban ghi vi sinh -> tui RONG. EmbeddingBag tra ve vector 0 cho tui rong.
_miss = BASE.loc[~BASE.study_id.isin(MICRO.study_id), ['study_id']].copy()
_miss['micro_org_ids'] = [[] for _ in range(len(_miss))]
_miss['micro_org_cnt'] = [[] for _ in range(len(_miss))]
_miss['micro_n_org_full'] = 0
MICRO = pd.concat([MICRO, _miss], ignore_index=True)
print(f'{len(MICRO):,} study | co organism {int((MICRO.micro_n_org_full > 0).sum()):,} '
      f'({(MICRO.micro_n_org_full > 0).mean():.1%}) | tui rong {int((MICRO.micro_n_org_full == 0).sum()):,}')

tu vung: 305 (gom <unk>) khop tren 71,387 cap TRAIN
so voi 29b: 304 organism thay vi 30  ->  10.1x

=== DUOI TU VUNG: bao nhieu organism hiem den muc nao (tren TRAIN) ===
  xuat hien o    1-   1 study:   47 organism  (      47 cap)
  xuat hien o    2-   4 study:   50 organism  (     142 cap)
  xuat hien o    5-  19 study:   88 organism  (     973 cap)
  xuat hien o   20-  99 study:   55 organism  (   2,712 cap)
  xuat hien o  100-  .. study:   64 organism  (  67,513 cap)

cap roi vao <unk> (organism khong co trong train): 167 (0.157%)
227,835 study | co organism 63,293 (27.8%) | tui rong 164,542


## C6 · Cổng, rồi ghi ra **file mới**

In [7]:
assert len(MICRO) == len(BASE) and MICRO.study_id.is_unique
assert set(MICRO.study_id) == set(BASE.study_id)
_bad = [i for ids in MICRO.micro_org_ids for i in ids if not (0 <= i < V)]
assert not _bad, f'id ngoai [0,{V}): {_bad[:10]}'
assert all(len(a) == len(b) for a, b in zip(MICRO.micro_org_ids, MICRO.micro_org_cnt)), \
    'do dai ids va cnt lech -> per_sample_weights se sai hang'
assert MICRO.micro_n_org_full.eq(MICRO.micro_org_ids.map(len)).all()
print('cong: khoa day du · id trong khoang · ids/cnt cung do dai · dem khop')

MICRO.to_parquet(P_MICRO, index=False)
json.dump({'V': V, 'id_to_name': VOCAB,
           'micro_past_d': MICRO_PAST_D, 'micro_future_h': MICRO_FUTURE_H,
           'fit_on': 'train only', 'unk_id': 0},
          open(P_VOCAB, 'w'), ensure_ascii=False, indent=1)
print(f'da ghi {P_MICRO}  ({os.path.getsize(P_MICRO)/1e6:.1f} MB)')
print(f'da ghi {P_VOCAB}  (V = {V:,})')
_chk = pd.read_parquet(P_MICRO)
assert _chk.micro_org_ids.map(len).equals(MICRO.micro_org_ids.map(len)), 'doc lai KHONG khop'
print('cong: doc lai khop — cot danh sach song sot qua parquet')

cong: khoa day du · id trong khoang · ids/cnt cung do dai · dem khop
da ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_micro_full.parquet  (1.5 MB)
da ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_micro_vocab.json  (V = 305)
cong: doc lai khop — cot danh sach song sot qua parquet


## C7 · `top-30` đã vứt đi bao nhiêu — đo trước khi ai kỳ vọng

Ba con số, và chúng nói ba chuyện khác nhau:
`độ phủ lần mọc` (top-30 bắt được bao nhiêu % sự kiện) · `study bị mất sạch danh tính`
(có vi khuẩn nhưng **không** cái nào trong top-30) · `study mất một phần`.

In [8]:
TOP30 = set(ORG_ALL.index[:ORG_TOP_K_REF])
res['in30'] = res.org_itemid.isin(TOP30)
print(f'=== TOP-{ORG_TOP_K_REF} cua 29b bo lai gi ===')
print(f'  do phu LAN MOC   : {res.in30.mean():.1%} cap nam trong top-{ORG_TOP_K_REF}')
_per = res.groupby('study_id').in30.agg(['sum', 'size'])
_all_out  = int((_per['sum'] == 0).sum())
_part_out = int(((_per['sum'] > 0) & (_per['sum'] < _per['size'])).sum())
_nstudy   = len(_per)
print(f'  study MAT SACH danh tinh : {_all_out:,} / {_nstudy:,} ({_all_out/_nstudy:.1%}) '
      f'-> chi con dem, khong con biet la con gi')
print(f'  study mat MOT PHAN       : {_part_out:,} / {_nstudy:,} ({_part_out/_nstudy:.1%})')
print(f'\n  29b in ra "phu ..% so lan mo tren train" — so nay phai KHOP voi dong dau.')

print(f'\n=== 20 organism dau NGOAI top-{ORG_TOP_K_REF} (theo so study TRAIN) ===')
print(ORG_ALL.iloc[ORG_TOP_K_REF:ORG_TOP_K_REF+20][['n_study', 'name']].to_string())
print(f'\n  DOC: neu `study MAT SACH danh tinh` nho (<5%) thi "lay het" chu yeu la don dep BIEU DIEN,')
print(f'       khong phai them thong tin — va dung ky vong mAP doi. Neu lon thi nguoc lai.')

=== TOP-30 cua 29b bo lai gi ===
  do phu LAN MOC   : 84.7% cap nam trong top-30
  study MAT SACH danh tinh : 4,675 / 63,293 (7.4%) -> chi con dem, khong con biet la con gi
  study mat MOT PHAN       : 8,500 / 63,293 (13.4%)

  29b in ra "phu ..% so lan mo tren train" — so nay phai KHOP voi dong dau.

=== 20 organism dau NGOAI top-30 (theo so study TRAIN) ===
            n_study                                             name
org_itemid                                                          
80191           336  HAEMOPHILUS INFLUENZAE, BETA-LACTAMASE POSITIVE
80007           331                           ENTEROBACTER AEROGENES
80228           324                                 CANDIDA GLABRATA
80055           318                      GRAM POSITIVE COCCUS(COCCI)
80172           311                            ENTEROCOCCUS FAECALIS
80248           305                     CITROBACTER FREUNDII COMPLEX
80289           302                            MORAXELLA CATARRHALIS
80108           2

## C8 · Phía mô hình — bản vá cho `EHRTok` / `EHRColTok`

**Chưa nối vào `37`/`38b`.** Đây là đặc tả để bước sau ráp, và nó **phải qua dry-run** trước khi
chạy thật — đúng kỷ luật đã dùng cho mọi thay đổi trước.

Ba chỗ phải sửa, và chỉ ba chỗ:

```
C1 cua 37/38b   nap cxrlt_micro_full.parquet -> hai tensor phang `flat` va `offs`
                VA BO 30 cot micro_org_* khoi ma tran dac (neu khong la dem HAI LAN)
EHRTok          ho `micro`: Linear(30->d)  ->  MicroBag(V, d)
EHRColTok       bag ra MOT token noi them, KHONG phai V token
```

```python
class MicroBag(nn.Module):
    '''Thay Linear(one-hot 30) bang EmbeddingBag tren TOAN BO tu vung.
       Linear(one-hot) = sum cot trong so cua cac id co mat = dung phep toan nay,
       nen day KHONG phai kien truc moi — chi la CUNG phep toan tren tu vung day du,
       va O(so id co mat) thay vi O(V).'''
    def __init__(s, V, d):
        super().__init__()
        s.bag = nn.EmbeddingBag(V, d, mode='sum', include_last_offset=True)
        nn.init.normal_(s.bag.weight, std=d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))   # khop dang cua EHRTok
    def forward(s, flat, offs, w):
        # tui RONG (study khong co ban ghi vi sinh) -> EmbeddingBag tra vector 0. Dung nhu mong doi.
        return s.mlp(s.bag(flat, offs, per_sample_weights=w))
```

**Cổng bắt buộc khi ráp:**

```
1. thay MicroBag ma GIU tu vung 30  ->  ket qua phai TRUNG voi ban Linear trong sai so may
   (neu lech thi loi la o day, khong phai o tu vung day du)
2. bo 30 cot micro_org_* khoi ma tran dac — dem hai lan la loi im lang, khong assert nao bat duoc
3. trao nhan qua van chay cho arm co micro
```

Cổng 1 là cổng quan trọng nhất: nó tách *"đổi cách cài đặt"* khỏi *"đổi từ vựng"*, để nếu số có
thay đổi thì biết là do cái nào.

In [9]:
print('C8 la DAC TA, khong chay gi. Buoc sau:')
print('  1. sua C1 cua 37/38b: nap', os.path.basename(P_MICRO), '-> flat/offs/w, BO cot micro_org_*')
print('  2. them MicroBag vao C3 (kien truc), noi vao ho `micro` cua EHRTok')
print('  3. dry-run voi cong 1 (tu vung 30 phai trung ban Linear) TRUOC khi chay that')
print(f'\n  V = {V:,} | file da san sang: {P_MICRO}')

C8 la DAC TA, khong chay gi. Buoc sau:
  1. sua C1 cua 37/38b: nap cxrlt_micro_full.parquet -> flat/offs/w, BO cot micro_org_*
  2. them MicroBag vao C3 (kien truc), noi vao ho `micro` cua EHRTok
  3. dry-run voi cong 1 (tu vung 30 phai trung ban Linear) TRUOC khi chay that

  V = 305 | file da san sang: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_micro_full.parquet
